# LC 496 — Next Greater Element I
**Day-71 | Monotonic Stack | Easy**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> For each element in <code>nums1</code>,
find its next greater element in <code>nums2</code>.
Pre-compute all next-greater answers for <code>nums2</code> using a
monotonic <em>decreasing</em> stack, store them in a hashmap,
then look up each element of <code>nums1</code> in O(1).
</div>

## Official Problem Statement

The **next greater element** of some element `x` in an array is the
first greater element that is to the right of `x` in the same array.

You are given two distinct 0-indexed integer arrays `nums1` and
`nums2`, where `nums1` is a subset of `nums2`.

For each `0 <= i < nums1.length`, find the index `j` such that
`nums1[i] == nums2[j]` and determine the next greater element of
`nums2[j]` in `nums2`. If there is no next greater element, the
answer for this query is `-1`.

Return an array `ans` of length `nums1.length` such that `ans[i]`
is the next greater element as described above.

**Example 1:**
```
Input:  nums1 = [4,1,2], nums2 = [1,3,4,2]
Output: [-1,3,-1]
```

**Example 2:**
```
Input:  nums1 = [2,4], nums2 = [1,2,3,4]
Output: [3,-1]
```

**Constraints:**
- `1 <= nums1.length <= nums2.length <= 1000`
- `0 <= nums1[i], nums2[i] <= 10^4`
- All integers in `nums1` and `nums2` are **unique**.
- All integers in `nums1` also appear in `nums2`.

## What This Is Actually Asking

For each value in `nums1`, go find that same value inside `nums2`,
then scan rightward in `nums2` until you hit something larger.
That larger value is the answer. If nothing larger exists to the
right, answer is -1.

```
nums1 = [4, 1, 2]
nums2 = [1, 3, 4, 2]

4 is at index 2 in nums2 → scan right → only 2 remains, 2 < 4 → -1
1 is at index 0 in nums2 → scan right → 3 > 1 ✓             →  3
2 is at index 3 in nums2 → scan right → nothing              → -1

Output: [-1, 3, -1]
```

Naive: O(m * n) — for each of m elements in nums1, scan up to n elements in nums2.
Smart: O(m + n) — precompute all next-greater answers for nums2 once,
then answer each nums1 query in O(1) via hashmap.

## Walk Through an Example by Hand

`nums2 = [1, 3, 4, 2]`

We want: `nge = { 1:3, 3:4, 4:-1, 2:-1 }`

```
stack = []   nge = {}

i=0, val=1
  stack empty → push 1       stack=[1]

i=1, val=3
  stack[-1]=1 < 3 → pop → nge[1]=3    stack=[]
  stack empty → push 3       stack=[3]

i=2, val=4
  stack[-1]=3 < 4 → pop → nge[3]=4    stack=[]
  stack empty → push 4       stack=[4]

i=3, val=2
  stack[-1]=4 > 2 → stop
  push 2                     stack=[4,2]

Loop ends — remaining stack has no next-greater:
  nge[4] = -1
  nge[2] = -1

nge = { 1:3, 3:4, 4:-1, 2:-1 }

Now answer nums1 = [4, 1, 2]:
  nge[4] = -1
  nge[1] =  3
  nge[2] = -1

Output: [-1, 3, -1]  ✓
```

## The Picture

Bar chart for `nums2 = [1, 3, 4, 2]`:

```
  4  |          ##
  3  |    ##    ##
  2  |    ##    ##    ##
  1  | ##  ##    ##    ##
      i=0  i=1  i=2  i=3
       1    3    4    2

  i=0 (1): next greater → 3  (one step right)
  i=1 (3): next greater → 4  (one step right)
  i=2 (4): next greater → ∅  (nothing taller to the right)
  i=3 (2): next greater → ∅  (end of array)
```

Stack trace (values, decreasing — stack is a waiting room):

```
  Push 1          stack=[1]
  3 arrives → 1 resolved → nge[1]=3   stack=[3]
  4 arrives → 3 resolved → nge[3]=4   stack=[4]
  Push 2          stack=[4,2]
  End of array → nge[4]=-1, nge[2]=-1
```

Key: Stack holds values **waiting** to be resolved.
A taller incoming bar resolves everyone shorter than itself.

## When To Use This Pattern

Use **monotonic stack + hashmap** when:
- You need the next greater (or smaller) element for values
  in one array, defined by their position in another array
- Queries are offline — you can precompute all answers before
  answering any query
- All values are unique (guarantees hashmap keys are clean)

**The key split:**
- Stack operates entirely on `nums2` — one pass, O(n)
- Hashmap stores `value → next_greater_value`
- `nums1` lookup is just `nge.get(x, -1)` per element

**Related problems:**
- LC 503 Next Greater Element II (circular array, double-pass)
- LC 739 Daily Temperatures (store distances, not values)
- LC 84 Largest Rectangle in Histogram (areas, not elements)

## The Approach

**Data structure:** Stack of **values** (monotonic decreasing).
Hashmap `nge`: value → its next greater element in `nums2`.

**Algorithm:**
1. `stack = []`, `nge = {}`
2. Iterate `val` through `nums2`:
   - While `stack` not empty AND `stack[-1] < val`:
     - `nge[stack.pop()] = val`
   - Push `val` onto stack
3. After loop — remaining stack has no next-greater:
   - For each leftover in stack: `nge[v] = -1`
4. Return `[nge[x] for x in nums1]`

**Why store values (not indices)?**  
All values are unique, so `value` is a safe hashmap key.
We don't need the index — we need the value itself to key
into `nge` when answering `nums1` queries.

**Complexity:** O(m + n) time, O(n) space.

In [1]:
from typing import List

In [2]:
# ── Test harness ─────────────────────────────────────────────────────

def test_harness(func):
    """
    Runs func(nums1, nums2) against known test cases.
    Prints PASSED / FAILED per case and a final summary.

    Args:
        func: callable(List[int], List[int]) -> List[int]
    """
    test_cases = [
        {
            "nums1":  [4, 1, 2],
            "nums2":  [1, 3, 4, 2],
            "expect": [-1, 3, -1],
            "label":  "Example 1 (LeetCode)",
        },
        {
            "nums1":  [2, 4],
            "nums2":  [1, 2, 3, 4],
            "expect": [3, -1],
            "label":  "Example 2 (LeetCode)",
        },
        {
            "nums1":  [1],
            "nums2":  [1, 2],
            "expect": [2],
            "label":  "Single query, answer exists",
        },
        {
            "nums1":  [2],
            "nums2":  [1, 2],
            "expect": [-1],
            "label":  "Single query, last element",
        },
        {
            "nums1":  [5, 4, 3, 2, 1],
            "nums2":  [1, 2, 3, 4, 5],
            "expect": [-1, 5, 4, 3, 2],
            "label":  "nums1 reverse of strictly increasing nums2",
        },
        {
            "nums1":  [1, 3, 5],
            "nums2":  [5, 4, 3, 2, 1],
            "expect": [-1, -1, -1],
            "label":  "Strictly decreasing nums2 — all -1",
        },
        {
            "nums1":  [3],
            "nums2":  [3],
            "expect": [-1],
            "label":  "Single element arrays",
        },
        {
            "nums1":  [1, 3],
            "nums2":  [3, 1, 2],
            "expect": [2, -1],
            "label":  "nums1 element appears before its greater in nums2",
        },
    ]

    passed = 0
    for tc in test_cases:
        result = func(tc["nums1"], tc["nums2"])
        ok = result == tc["expect"]
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(f"  [{status}] {tc['label']}")
        if not ok:
            print(f"    got:      {result}")
            print(f"    expected: {tc['expect']}")

    total = len(test_cases)
    print(f"\n  Summary: {passed}/{total} passed")


print("Test harness defined.")

Test harness defined.


In [12]:
from typing import List

def next_greater_element(nums1: List[int], nums2: List[int]) -> List[int]:
    """
    LC 496 — Next Greater Element I
    Approach: Monotonic decreasing stack over nums2.
    Build a hashmap nge: value -> next greater value.
    Answer each nums1 query via nge lookup in O(1).
    Args:
        nums1 (List[int]): query array, subset of nums2.
        nums2 (List[int]): source array, all unique values.
    Returns:
        List[int]: next greater element in nums2 for each
            value in nums1, or -1 if none exists.
    Time:  O(m + n)  — one pass over nums2, one pass over nums1
    Space: O(n)      — stack + hashmap, both at most size n
    """
    stack = [-1]  * len(nums1) # waiting room — numbers sitting here haven't found their NGE yet
    nge   = {}  # answer map  — nge[x] = first number to the right of x that beats it
    res   = []

    for num in nums2:
        # new number arrived — evict anyone in the waiting room it beats
        while stack and stack[-1] < num:
            nge[stack.pop()] = num   # num is the NGE for the evicted value
        stack.append(num)            # num joins the waiting room

    # anyone still waiting never got beaten — no NGE exists
    while stack:
        nge[stack.pop()] = -1

    # answer each nums1 query in O(1) via the map we just built
    for num in nums1:
        res.append(nge[num])

    return res


print(next_greater_element([4,1,2], [1,3,4,2]))   # [-1, 3, -1]
print(next_greater_element([2,4],   [1,2,3,4]))   # [3, -1]
# test_harness(next_greater_element)
print("next_greater_element defined.")

[-1, 3, -1]
[3, -1]
next_greater_element defined.


In [ ]:
def next_greater_element(nums1: List[int], nums2: List[int]) -> List[int]:
    """
    LC 496 — Next Greater Element I

    Approach: Monotonic decreasing stack over nums2.
    Build a hashmap nge: value -> next greater value.
    Answer each nums1 query via nge lookup in O(1).

    Args:
        nums1 (List[int]): query array, subset of nums2.
        nums2 (List[int]): source array, all unique values.

    Returns:
        List[int]: next greater element in nums2 for each
            value in nums1, or -1 if none exists.

    Time:  O(m + n)  — one pass over nums2, one pass over nums1
    Space: O(n)      — stack + hashmap, both at most size n
    """
r'''
'''


print("next_greater_element shell defined.")

In [ ]:
# Quick debug — run while building
# print(next_greater_element([4,1,2], [1,3,4,2]))   # [-1, 3, -1]
# print(next_greater_element([2,4],   [1,2,3,4]))   # [3, -1]

# Uncomment when solution is ready
# test_harness(next_greater_element)

## Complexity

| | Time | Space |
|---|---|---|
| Overall | **O(m + n)** | **O(n)** |

**Time breakdown:**
- One pass over `nums2` (length n): each value pushed and popped at most once → O(n).
- One pass over `nums1` (length m): one hashmap lookup per element → O(m).
- Total: O(m + n).

**Space breakdown:**
- `stack`: O(n) worst case — strictly decreasing `nums2` means nothing pops until end.
- `nge` hashmap: O(n) — one entry per element of `nums2`.
- Output array: O(m).

**Compared to brute force:**
- Brute force: O(m × n) — for each of m queries, scan up to n elements.
- Stack approach: O(m + n) — precompute once, answer each query in O(1).

## Real World Connection

**Alert escalation in monitoring pipelines.**

Imagine a time-series of server CPU readings (`nums2`). A second
list (`nums1`) contains specific readings flagged for review.
For each flagged reading, the on-call engineer wants to know:
"What is the next spike higher than this one?" — so they can
estimate how quickly the situation escalated.

Pre-computing all next-greater values once and storing them
in a lookup table is exactly this problem.

**Other domains:**
- **Stock screening:** given a watchlist subset of tickers,
  find the next trading day each ticker exceeded a prior close.
- **ETL pipelines:** for a filtered set of events, find the
  next higher-priority event in the full event log.
- **Game leaderboards:** for a set of players, find the next
  score above theirs in the global ranking.

The hashmap lookup pattern (precompute offline, answer online)
is a fundamental technique in data engineering — trade space
for repeated query speed.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra